In [1]:
import sys
import time
import torch
import torchvision
from pathlib import Path
import os
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from visdrone_toolkit import VisDroneDataset
from visdrone_toolkit.utils import collate_fn
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchmetrics.detection import MeanAveragePrecision
from torchvision.ops import box_iou

C:\Users\Ray\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_img = r"D:\cv\Dataset\VisDrone2019-DET-train\images"
train_annotations =r"D:\cv\Dataset\VisDrone2019-DET-train\annotations"

val_img = r"D:\cv\Dataset\VisDrone2019-DET-val\images"
val_annotations= r"D:\cv\Dataset\VisDrone2019-DET-val\annotations"

test_img = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\images"
test_annotations = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\annotations"





In [3]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")

weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT

model = retinanet_resnet50_fpn(
    weights=weights,
)

device = torch.device("cuda")
model = model.to(device)





In [4]:
model.load_state_dict(
    torch.load(
        r"D:\cv\models\retinanet_visdrone_100ep.pth",
        map_location=device,
        weights_only=True
    )
)

model = model.to(device)
model.eval()

RetinaNet(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d(256, eps=0.0)


In [5]:
val_dataset = VisDroneDataset(
    image_dir=test_img,
    annotation_dir=test_annotations,
    filter_ignored=True,
    filter_crowd=True,
)


Found 1610 images in D:\cv\Dataset\VisDrone2019-DET-test-dev\images


In [6]:
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_fn,
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0025,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=450
)

scaler = torch.amp.GradScaler("cuda")



In [7]:
metric = MeanAveragePrecision(
    box_format="xyxy",
    iou_type="bbox"
)

TP = 0
FP = 0
FN = 0

CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5

In [8]:
with torch.inference_mode():

    progress_bar = tqdm(
        val_loader,
        desc="Evaluation",
        unit="batch",
        dynamic_ncols=True
    )

    for images, targets in progress_bar:

        images = [
            image.to(device, non_blocking=True)
            for image in images
        ]

        predictions = model(images)

        # --------------------------------------------------
        # mAP
        # --------------------------------------------------

        metric.update(
            [
                {
                    "boxes": prediction["boxes"].cpu(),
                    "scores": prediction["scores"].cpu(),
                    "labels": prediction["labels"].cpu(),
                }
                for prediction in predictions
            ],
            [
                {
                    "boxes": target["boxes"],
                    "labels": target["labels"],
                }
                for target in targets
            ]
        )

        # --------------------------------------------------
        # TP / FP / FN
        # --------------------------------------------------

        for prediction, target in zip(predictions, targets):

            pred_boxes = prediction["boxes"].cpu()
            pred_scores = prediction["scores"].cpu()
            pred_labels = prediction["labels"].cpu()

            gt_boxes = target["boxes"]
            gt_labels = target["labels"]

            # Confidence threshold
            keep = pred_scores >= CONF_THRESHOLD

            pred_boxes = pred_boxes[keep]
            pred_scores = pred_scores[keep]
            pred_labels = pred_labels[keep]

            # Обрабатываем сначала самые уверенные detections
            order = pred_scores.argsort(descending=True)

            pred_boxes = pred_boxes[order]
            pred_labels = pred_labels[order]

            matched_gt = torch.zeros(
                len(gt_boxes),
                dtype=torch.bool
            )

            for pred_box, pred_label in zip(
                pred_boxes,
                pred_labels
            ):

                if len(gt_boxes) == 0:
                    FP += 1
                    continue

                # GT только того же класса
                class_mask = gt_labels == pred_label

                candidate_indices = torch.where(
                    class_mask & ~matched_gt
                )[0]

                if len(candidate_indices) == 0:
                    FP += 1
                    continue

                ious = box_iou(
                    pred_box.unsqueeze(0),
                    gt_boxes[candidate_indices]
                )[0]

                best_iou, best_idx = ious.max(dim=0)

                if best_iou >= IOU_THRESHOLD:

                    gt_idx = candidate_indices[best_idx]

                    matched_gt[gt_idx] = True
                    TP += 1

                else:
                    FP += 1

            FN += (~matched_gt).sum().item()


# ============================================================
# Итоговые метрики
# ============================================================

results = metric.compute()

mAP_50 = results["map_50"].item()
mAP_50_95 = results["map"].item()

precision = (
    TP / (TP + FP)
    if TP + FP > 0
    else 0.0
)

recall = (
    TP / (TP + FN)
    if TP + FN > 0
    else 0.0
)


print("\n" + "=" * 55)
print("Faster R-CNN — VisDrone Evaluation")
print("=" * 55)

print(f"mAP@0.5:       {mAP_50:.4f}")
print(f"mAP@0.5:0.95:  {mAP_50_95:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"TP:             {TP}")
print(f"FP:             {FP}")
print(f"FN:             {FN}")

print("=" * 55)

Evaluation:   0%|          | 0/202 [00:00<?, ?batch/s]C:\Users\Ray\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
Evaluation: 100%|██████████| 202/202 [01:30<00:00,  2.24batch/s]



Faster R-CNN — VisDrone Evaluation
mAP@0.5:       0.1793
mAP@0.5:0.95:  0.0998
Precision:      0.4956
Recall:         0.3429
TP:             25749
FP:             26210
FN:             49353
